# 03_question_C
Retrieval cells run only when RETRIEVE is True; the last cell recomputes every logged count from the saved raw files.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('../env'))
import stir_l4_kit as k
k.use(os.path.abspath('..'))
RETRIEVE = False

In [ ]:
if RETRIEVE:
    k.mesh_lookup('SGLT2 inhibitors')

In [ ]:
if RETRIEVE:
    k.mesh_lookup('adults')

In [ ]:
if RETRIEVE:
    k.mesh_lookup('diabetes mellitus')

In [ ]:
if RETRIEVE:
    k.mesh_lookup('Asian people')

In [ ]:
if RETRIEVE:
    k.mesh_lookup('weight loss')

In [ ]:
if RETRIEVE:
    k.mesh_lookup('body fat')

In [ ]:
if RETRIEVE:
    k.mesh_lookup('skeletal muscle')

In [ ]:
if RETRIEVE:
    k.mesh_lookup('body water')

In [ ]:
if RETRIEVE:
    k.mesh_lookup('body composition')

In [ ]:
if RETRIEVE:
    k.mesh_lookup('placebo')

In [ ]:
if RETRIEVE:
    k.mesh_lookup('usual care')

In [ ]:
if RETRIEVE:
    k.mesh_lookup('no treatment')

In [ ]:
if RETRIEVE:
    k.mesh_lookup('randomized controlled trial')

In [ ]:
if RETRIEVE:
    k.mesh_lookup('cohort studies')

In [ ]:
if RETRIEVE:
    k.mesh_lookup('lean body mass')

In [ ]:
if RETRIEVE:
    k.mesh_lookup('muscle strength')

In [ ]:
if RETRIEVE:
    k.mesh_lookup('physical performance')

In [ ]:
if RETRIEVE:
    k.mesh_lookup('placebos')

In [ ]:
if RETRIEVE:
    k.mesh_lookup('fat-free mass')

## Analysis (session S2; reads saved files only)
Step 6, Research Question. Rule step6-mesh-1 chooses the heading for each part; rule step6-rule-1 counts comparisons and study types. Functions in env/stir_l4_local.py; outputs in data/derived/step6_mesh_headings.csv and data/derived/step6_comparators.csv.

In [ ]:
import stir_l4_local as L
# Medical Subject Headings for each part of the question (S2, Step 6), from the saved esummary responses above.
# Terms are Uday's words from the certified objective and focus; 'skeletal muscle' and 'body fat' disambiguate
# 'muscle' and 'fat', and 'body composition' (the three parts together) is the assistant's suggestion.
MESH = [("Intervention", "SGLT2 inhibitors", "L000308"), ("Population", "adults", "L000310"),
        ("Population (exclusion)", "diabetes mellitus", "L000312"), ("Subgroup", "Asian people", "L000314"),
        ("Outcome (weight lost)", "weight loss", "L000316"), ("Outcome (fat)", "body fat", "L000318"),
        ("Outcome (muscle)", "skeletal muscle", "L000320"), ("Outcome (body water)", "body water", "L000322"),
        ("Outcomes together (my suggestion)", "body composition", "L000324")]
mesh = L.mesh_headings(MESH)
for r in mesh["rows"]:
    print(r["term"], "->", r["heading"], "|", r["narrower_returned"])


In [ ]:
import stir_l4_local as L
# Comparisons and study types (S2, Step 6), rule step6-rule-1: the records of the trials search (L000195) tagged as
# randomized controlled trials, and the publication types of the broader search (L000193); records fetched in L000196.
comp = L.comparators_and_designs('L000196', 'L000195', 'L000193')
print({k_: v for k_, v in comp.items() if k_ != 'rows'})


## Analysis (session S3; reads saved files only)
Passages shown with the three open Step 6 questions, checked against the saved abstracts; output in data/derived/step6_passages_S3.csv.

In [ ]:
import stir_l4_local as L
# Session S3, Step 6: passages quoted with the three open questions (comparison, study types, what counts as muscle).
# Each must be a substring of the saved abstract after the declared normalization (L._norm_ws); reads saved files only.
# Records: L000196 (PubMed records of the Feasibility searches L000193 and L000195), L000129 and L000084 (review records).
S3_PASSAGES = [
    ("27550386", "dapagliflozin 10 mg once daily plus subcutaneous long-acting exenatide 2 mg once weekly or placebo",
     "comparison: the drug given with a second drug, against placebo"),
    ("39361103", "the combined use of empagliflozin (EMPA) and topiramate (TPM) versus a placebo",
     "comparison: the drug given with a second drug, against placebo"),
    ("42892668", "lean-mass estimates do not directly quantify contractile skeletal muscle",
     "muscle: lean mass is not muscle itself"),
    ("42892668", "may be influenced by alterations in body water", "muscle: lean mass and body water"),
    ("42892668", "insufficient to infer clinically meaningful muscle loss without concurrent assessment of muscle strength",
     "muscle: strength or performance"),
    ("42591093", "Primary outcomes were skeletal muscle mass (SMM) and lean mass (LM)",
     "muscle: the two kept apart (type 2 diabetes)"),
    ("42591093", "No SGLT2 inhibitor significantly reduced SMM compared with placebo",
     "muscle: the two kept apart (type 2 diabetes)"),
    ("40707349", "Physical-performance-tests remained unchanged under SGLT2i", "muscle: strength or performance"),
]
chk = L.check_passages(S3_PASSAGES, ["L000196", "L000129", "L000084"], out="data/derived/step6_passages_S3.csv")
print(chk["all_found"], [(r["pmid"], r["found_in_abstract"], r["source_log_id"]) for r in chk["rows"]])


In [ ]:
# Log the passage check (session S3)
lid_passages = k.log("computation", element="RQ", derived_path=chk["path"], refers_to=["L000196", "L000129", "L000084"],
    text=f"Step 6, session S3: {len(chk['rows'])} passages quoted with the open questions checked against the saved abstracts "
         f"({'all found' if chk['all_found'] else 'not all found'}); declared normalization: hyphenated line breaks joined, whitespace collapsed",
    parameters={"rule": "passage-check (L._norm_ws)", "all_found": chk["all_found"],
                "pmids": [r["pmid"] for r in chk["rows"]]})
print(lid_passages)


## Analysis (session S3, after Uday's answers on comparison, study types and muscle; reads saved files only)
Rule step6-mesh-1 for the new parts; rule step6-rule-2 for the criteria the answers leave open (second drug, people with diabetes in the study, prediabetes, treatment length, measuring method); passage check of every quote shown. Outputs in data/derived/step6_mesh_headings_S3.csv, step6_criteria_S3.csv, step6_methods_S3.csv and step6_passages_S3b.csv.

In [ ]:
import stir_l4_local as L
# Medical Subject Headings for the new parts (session S3). Uday's words: placebo, usual care, no treatment, randomized
# controlled trial (his 'rcts'), muscle strength and physical performance (his answer 3 d); 'placebos' and 'cohort studies'
# are my wording of 'placebo' and 'controlled cohorts'. 'lean body mass' (L000347) and 'fat-free mass' (L000354) returned no
# MeSH record, so they have no esummary to read.
MESH_S3 = [("Comparison", "placebo", "L000338"), ("Comparison", "placebos", "L000353"),
           ("Comparison", "usual care", "L000340"), ("Comparison", "no treatment", "L000342"),
           ("Study type", "randomized controlled trial", "L000344"), ("Study type", "cohort studies", "L000346"),
           ("Outcome (muscle)", "muscle strength", "L000349"), ("Outcome (muscle)", "physical performance", "L000351")]
mesh_s3 = L.mesh_headings(MESH_S3, out="data/derived/step6_mesh_headings_S3.csv")
for r in mesh_s3["rows"]:
    print(r["term"], "->", r["heading"], r["ui"], "|", r["tree_numbers"])
no_record = {t: k.records(kind="query")[[x["id"] for x in k.records(kind="query")].index(lid)]["n_total"]
             for t, lid in (("lean body mass", "L000347"), ("fat-free mass", "L000354"))}
print(mesh_s3["log_id"], no_record)

In [ ]:
import stir_l4_local as L
# Rule step6-rule-2 (session S3): criteria the answers leave open, over the saved records (L000196) of the trials search
# (L000195, records tagged 'Randomized Controlled Trial') and of the broader search (L000193). Treatment length is read
# from one declared passage per record (my reading of each abstract; the code checks it is in the saved abstract).
DURATIONS = {
    "42286641": "randomised to 180 days treatment with empagliflozin 10 mg or placebo",
    "39884573": "10 mg dapagliflozin, once daily for 24 weeks",
    "39361103": "(as mentioned for topiramate tablet in group 1), for 12 weeks",
    "38536017": "empagliflozin 10 mg daily or placebo (1:1 ratio) for 52 weeks",
    "38295394": "empagliflozin or matching placebo (2:1 ratio) for 12 weeks",
    "36592688": "underwent 2-weeks of treatment with dapagliflozin (10 mg/day) or placebo",
    "35478433": "we determined the effects of 6 weeks of dapagliflozin on markers of volume status",
    "34325888": "Patients underwent cardiac magnetic resonance at baseline and after 6 months",
    "34097062": "PHEN (7.5 mg)/TPM (46 mg ER daily) treatment for 24 weeks",
    "33357505": "empagliflozin 10 mg or matched placebo once daily for 12 weeks",
    "32975679": "or placebo (n = 47) for 24 weeks",
    "28345814": "They entered an open-label extension from 24 to 52 weeks during which all participants received active treatment",
    "27550386": "completed 24 weeks of treatment",
}
crit = L.criteria_facts("L000196", "L000195", "L000193", DURATIONS)
print({k_: v for k_, v in crit.items() if k_ not in ("rows", "method_rows")})
for r in crit["rows"]:
    print(r["pmid"], r["weeks"], r["duration_found"], r["mixed_diabetes"], r["names_prediabetes"], r["second_drug_in_title"])
for r in crit["method_rows"]:
    print(r["pmid"], r["method"], r["rct_tag"], r["sentence"][:120])

In [ ]:
import stir_l4_local as L
# Rule step6-rule-3 (supersedes the rule step6-rule-2 computations L000356 and L000357, the second a re-run with the same
# result, before anything was shown): insulin resistance alone no longer counts as prediabetes; record 39884573 names it as a
# result, not as who was enrolled. Same passages (DURATIONS, cell above).
crit3 = L.criteria_facts("L000196", "L000195", "L000193", DURATIONS, rule=L.RULE_6_3, prediab=L.PREDIAB_3,
                         out="data/derived/step6_criteria_S3_r3.csv", out_methods="data/derived/step6_methods_S3.csv")
corr = k.log("correction", refers_to=["L000356", "L000357"],
             text=f"Superseded by {crit3['log_id']} (rule step6-rule-3) before they were shown: insulin resistance alone no "
                  "longer counts as prediabetes (record 39884573 names it as a result); L000357 re-ran L000356 with the same result.")
print({k_: v for k_, v in crit3.items() if k_ not in ("rows", "method_rows")}, corr)

In [ ]:
import stir_l4_local as L
# Session S3, Step 6 (after Uday's answers): passages quoted with the draft question and the three open criteria, checked
# against the saved abstracts (L000196) after the declared normalization (L._norm_ws).
S3B_PASSAGES = [
    ("34325888", "in addition to optimal medical treatment", "second drug: background treatment in both groups"),
    ("27550386", "mostly attributable to adipose tissue reduction without lean tissue change",
     "second drug: a combination trial reports the parts of the loss"),
    ("38295394", "Thirty individuals with T2D and 27 without were randomly assigned", "people with and without diabetes"),
    ("33357505", "including 105 (88%) without diabetes", "people with and without diabetes: share"),
    ("28345814", "73.5% of participants had prediabetes", "prediabetes"),
]
chk_b = L.check_passages(S3B_PASSAGES, ["L000196"], out="data/derived/step6_passages_S3b.csv")
lid_b = k.log("computation", element="RQ", derived_path=chk_b["path"], refers_to=["L000196"],
    text=f"Step 6, session S3 (after Uday's answers): {len(chk_b['rows'])} passages quoted with the draft question checked "
         f"against the saved abstracts ({'all found' if chk_b['all_found'] else 'not all found'}); declared normalization: "
         "hyphenated line breaks joined, whitespace collapsed",
    parameters={"rule": "passage-check (L._norm_ws)", "all_found": chk_b["all_found"],
                "pmids": [r["pmid"] for r in chk_b["rows"]]})
print(chk_b["all_found"], [(r["pmid"], r["found_in_abstract"]) for r in chk_b["rows"]], lid_b)

## Analysis (session S3, after Uday's choices 1 a, 2 b, 3 a; reads saved files only)
Passages quoted with the question shown for certification, checked against the saved abstracts (L000196). The counts shown with them (3 records with a second drug, 3 enrolling people with and without diabetes, 13 trial records, 2 under 12 weeks) are those of rule step6-rule-3 above; nothing new is counted.

In [ ]:
import stir_l4_local as L
# Session S3, Step 6 (question for certification): what Uday's choices keep in or leave out, read from the saved
# abstracts. The other group in each record with a second drug gets placebo alone; one trial with several groups gives
# exenatide with and without dapagliflozin (my suggestion: a pair of groups within a larger trial counts if it fits).
S3C_PASSAGES = [
    ("39361103", "Participants received an empagliflozin placebo (daily) plus a topiramate placebo",
     "choice 1 a: second drug, the other group gets placebo alone"),
    ("27550386", "plus subcutaneous long-acting exenatide 2 mg once weekly or placebo",
     "choice 1 a: second drug, the other group gets placebo alone"),
    ("28345814", "plus subcutaneous long-acting exenatide 2 mg once weekly or to placebo",
     "choice 1 a: second drug, the other group gets placebo alone"),
    ("34097062", "exenatide (EQW) and dapagliflozin (DAPA) alone and coadministered (EQW/DAPA)",
     "suggestion: a pair of groups within a larger trial"),
]
chk_c = L.check_passages(S3C_PASSAGES, ["L000196"], out="data/derived/step6_passages_S3c.csv")
lid_c = k.log("computation", element="RQ", derived_path=chk_c["path"], refers_to=["L000196", "L000358"],
    text=f"Step 6, session S3 (question for certification): {len(chk_c['rows'])} passages read for what Uday's choices "
         f"keep in or leave out, checked against the saved abstracts ({'all found' if chk_c['all_found'] else 'not all found'}); "
         "declared normalization: hyphenated line breaks joined, whitespace collapsed",
    parameters={"rule": "passage-check (L._norm_ws)", "all_found": chk_c["all_found"],
                "pmids": [r["pmid"] for r in chk_c["rows"]]})
print(chk_c["all_found"], [(r["pmid"], r["found_in_abstract"]) for r in chk_c["rows"]], lid_c)

In [ ]:
print(k.verify()['md'])